In [ ]:
# load the airports 
import pandas as pd
airports = pd.read_csv("airports.csv")
# drop all wirports, that are not in the US or Puerto Rico or Virgin Islands
# airports = airports[airports['iso_country'] == 'US']
# columns to drop
drop_cols = ["keywords", "wikipedia_link", "home_link", "continent","gps_code", "local_code"]
airports = airports.drop(columns=drop_cols)
display(airports)

In [ ]:
# load the small flight data
flights = pd.read_feather("all_flights.feather")
# get the unique Origin airports
unique_airports = flights['Origin'].unique()
display(unique_airports)

# filter the airports dataframe to only include the airports that are in the flights data
airports = airports[airports['iata_code'].isin(unique_airports)]
display(airports)
# show the airports that are not in the airports dataframe
missing_airports = set(unique_airports) - set(airports['iata_code'])
print("Missing airports:", missing_airports)


In [ ]:
# load the runways data
runways = pd.read_csv("runways.csv")
# drop all runways, that are not in the US or Puerto Rico or Virgin Islands
# list wiuth all different surface types

#drop all rows whos airport_ref is not in the airports dataset id column
runways = runways[runways['airport_ref'].isin(airports['id'])]
# display(runways)
# get for airport the number of runways and the most common surface type, and the average length of the runways, if all the runays have lightning
runway_info = runways.groupby('airport_ref').agg({'id': 'count', 'surface': lambda x: x.mode()[0], 'length_ft': 'mean','lighted': 'sum'}).reset_index()
runway_info.columns = ['airport_ref', 'num_runways', 'most_common_surface', 'avg_runway_length', 'num_lighted_runways']
# display(runway_info)
#get the unique values of the most common surface type


# change all values that start with ASP to ASPHALT and all values that start with CONC to CONCRETE
runway_info['most_common_surface'] = runway_info['most_common_surface'].apply(lambda x: 'ASPHALT' if str(x).startswith('ASP') else ('CONCRETE' if str(x).startswith('CON') else x))
# display(runway_info)
#

# join this runway info with the airports dataframe on the id and airport_ref columns
airports = airports.merge(runway_info, left_on='id', right_on='airport_ref', how='left')
display(airports)

In [ ]:
# drop all columns that are not needed for the model building
airports = airports.drop(columns=['id', 'airport_ref',"ident"])
# change the sheduled_service from yes not to a boolean value
airports['scheduled_service'] = airports['scheduled_service'].map({'yes': True, 'no': False})
# chnage the lighted_runnways to 1 if runways -lighted runways is 0, 0.5 if it is more than 0 and 0 if the number of lighted runways is 0
airports['num_lighted_runways'] = airports.apply(lambda row: 1 if row['num_runways'] - row['num_lighted_runways'] == 0 else (0.5 if row['num_lighted_runways'] > 0 else 0), axis=1)
# chnage name of this collumn to has_lighted_runways
airports = airports.rename(columns={'num_lighted_runways': 'has_lighted_runways'})
display(airports)

In [ ]:
# load the airports only data
airports_only = pd.read_csv("Airports-Only.csv",delimiter=",", encoding='latin-1')
display(airports_only)
# only keep IATA Timezone DST and TZ
airports_only = airports_only[["IATA", "Timezone", "DST", "TZ"]]
# join this with the airports dataframe on the iata_code and IATA columns
airports = airports.merge(airports_only, left_on='iata_code', right_on='IATA', how='left')
# drop the IATA column
airports = airports.drop(columns=['IATA'])
display(airports)

In [ ]:
airport_limit_list =["JFK","LAX","MIA","SFO","EWR","ORD","ATL","DFW","IAH",
"BOS","MCO","FLL","SEA","CLT","DEN","PHL","LAS","HNL","DTW","MSP","PHX","LGA","TPA",
"SLC","BWI","AUS","SAN","HOU","PDX","MDW","OAK","BNA","DCA","STL","DAL"]

In [ ]:
# import meteostat
import meteostat as ms
import datetime
import numpy as np
inventory_start = datetime.date(2009, 1, 1)
inventory_end = datetime.date(2019, 12, 31)
dummy_params =["temp","prcp","wspd"]
extended_params = ["temp","prcp","wspd","coco","cldc","tsun","wpgt"]
# for every aiport find the 3 closest weather stations and get their ids
airports['airport_station'] = None
airports['airport_station_distance'] = np.nan
airports['airport_station_name'] = None
airports['airport_weather_params'] = np.nan
airports['airport_data_length_code'] = np.nan
for i in range(1, 4):
    airports[f'closest_station_{i}'] = None
    airports[f'closest_station_{i}_distance'] = np.nan
    airports[f'closest_station_{i}_name'] = None
for index, row in airports.iterrows():
    airport_location = ms.Point(row['latitude_deg'], row['longitude_deg'])
    waether_stations = None
    try:
        # get the nearby stations for the airport
        waether_stations = ms.stations.nearby(airport_location, limit=6, radius=50000)
    except Exception as e:
        print(f"Error occurred while fetching nearby stations for airport {row['Name']}")
        continue
    # try to find the airport's own 

    for idx, station in waether_stations.iterrows():
        station_meta = ms.stations.meta(station.name)
        station_airport_code = station_meta.identifiers.get("iata", "N/A")
        station_airfield_code = station_meta.identifiers.get("icao", "N/A")
        if station_airport_code == row["iata_code"] or station_airfield_code == row["icao_code"]:
            # print(f"Checking station {station.name} with IATA code {station_airport_code} for airport {row['iata_code']}")
            airports.loc[index, 'airport_station'] = station.name
            airports.loc[index, 'airport_station_distance'] = station['distance']
            airports.loc[index, 'airport_station_name'] = station_meta.name
            # remove this station from the stations dataframe
            waether_stations = waether_stations[waether_stations.name != station.name]
            inventory = ms.stations.inventory(station.name)
            # further check if the airport is in the airport_limit_list, if it is not we will only check for the dummy parameters, if it is we will check for the extended parameters
            if row['iata_code'] in airport_limit_list:
                available_params = []
                for param in extended_params:
                    if param in inventory.parameters:
                        available_params.append(param)
                        
                print(f"Airport {row['iata_code']} has the following available parameters: \n {available_params}")

            if all(param in inventory.parameters for param in dummy_params):
                 airports.loc[index, 'airport_weather_params'] = 1
            elif any(param in inventory.parameters for param in dummy_params):
                airports.loc[index, 'airport_weather_params'] = 0.5
            else:
                airports.loc[index, 'airport_weather_params'] = np.nan
            if inventory_start >= inventory.start and inventory_end <= inventory.end:
                airports.loc[index, 'airport_data_length_code'] = 1
            # check if there is at least some data in the inventory for the time period we are interested in
            elif inventory_start <= inventory.end and inventory_end >= inventory.start:
                airports.loc[index, 'airport_data_length_code'] = 0.5
            else:
                airports.loc[index, 'airport_data_length_code'] = np.nan
            break

    # get the 3 closest stations to the airport
    closest_stations = waether_stations.head(3)
    # # get the stations ids and the distance to the airport
    for station_idx, (idx, station) in enumerate(closest_stations.iterrows(), start=1):
        station_meta = ms.stations.meta(station.name)
        distance = station['distance']
        # print(station)
        airports.loc[index, f'closest_station_{station_idx}'] = str(station.name)
        airports.loc[index, f'closest_station_{station_idx}_distance'] = distance
        airports.loc[index, f'closest_station_{station_idx}_name'] = str(station_meta.name)
display(airports)

In [ ]:
# save the airports dataframe to a new csv file
airports.to_csv("airports_with_runway_info.csv", index=False)